In [1]:
from statsmodels.tsa.api import VAR
import pandas as pd
import numpy as np

In [2]:
mat_list = ["0Y1M", "0Y3M", "0Y6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]

df = pd.read_csv("../../data/FRB_H15.csv").dropna()
df.columns = ["Date", *mat_list]
df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date", inplace=True)

In [3]:
# Estimate the three Nelson-Siegel beta factors for each yield curve.
maturities = [1/12, 3/12, 6/12, 1, 2, 3, 5, 7, 10, 20, 30]
lambda_ns = 0.0609
loadings = np.array([
    [
        1,
        (1 - np.exp(-lambda_ns * maturity)) / (lambda_ns * maturity),
        ((1 - np.exp(-lambda_ns * maturity)) / (lambda_ns * maturity))
        - np.exp(-lambda_ns * maturity),
    ]
    for maturity in maturities
])

beta_values = [
    np.linalg.lstsq(loadings, yields, rcond=None)[0]
    for yields in df.to_numpy()
]
betas = pd.DataFrame(
    beta_values,
    index=df.index,
    columns=["Beta 1", "Beta 2", "Beta 3"],
)

In [4]:
# Fit a five-lag VAR and make rolling one-day forecasts over the final 20 observations.
test_size = 20
lag = 5
train_betas = betas.iloc[:-test_size]
test_betas = betas.iloc[-test_size:]

var_result = VAR(train_betas.reset_index(drop=True)).fit(lag)
history = train_betas.to_numpy().tolist()
forecasts = []

for actual_betas in test_betas.to_numpy():
    forecast = var_result.forecast(
        np.asarray(history[-var_result.k_ar:]),
        steps=1,
    )[0]
    forecasts.append(forecast)
    history.append(actual_betas)

forecast_betas = pd.DataFrame(
    forecasts,
    index=test_betas.index,
    columns=betas.columns,
)

In [5]:
beta_rmses = np.sqrt(
    np.mean((test_betas.to_numpy() - forecast_betas.to_numpy()) ** 2, axis=0)
)

for beta, beta_rmse in enumerate(beta_rmses, start=1):
    print(f"  Beta: {beta}")
    print(f"  Forecast 1 day, RMSE: {beta_rmse}")

  Beta: 1
  Forecast 1 day, RMSE: 0.2620595911243325
  Beta: 2
  Forecast 1 day, RMSE: 0.273142624962925
  Beta: 3
  Forecast 1 day, RMSE: 0.5013297827734863
